In [1]:
import psycopg2
import pandas as pd

In [2]:
conn=psycopg2.connect(password="jaune2000",user="postgres",port="5432",host="localhost",database="megabase0")
cur=conn.cursor()
conn.autocommit=True

In [6]:
cur.execute("EXPLAIN ANALYSE SELECT sum(nb) FROM entrepot.fait_etablissement WHERE insee_code = '69123'")
cur.fetchall()

[('Aggregate  (cost=11.69..11.70 rows=1 width=8) (actual time=3.172..3.174 rows=1.00 loops=1)',),
 ('  Buffers: shared read=6',),
 ('  ->  Bitmap Heap Scan on fait_etablissement  (cost=4.30..11.69 rows=2 width=4) (actual time=2.017..2.754 rows=4.00 loops=1)',),
 ("        Recheck Cond: ((insee_code)::text = '69123'::text)",),
 ('        Heap Blocks: exact=4',),
 ('        Buffers: shared read=6',),
 ('        ->  Bitmap Index Scan on fait_etablissement_pkey  (cost=0.00..4.30 rows=2 width=0) (actual time=0.434..0.435 rows=4.00 loops=1)',),
 ("              Index Cond: ((insee_code)::text = '69123'::text)",),
 ('              Index Searches: 1',),
 ('              Buffers: shared read=2',),
 ('Planning:',),
 ('  Buffers: shared hit=71 read=2',),
 ('Planning Time: 4.217 ms',),
 ('Execution Time: 3.303 ms',)]

In [19]:
cur.execute("EXPLAIN ANALYSE SELECT sum(nb) FROM entrepot.fait_etablissement WHERE type = 'lycee'")
cur.fetchall()

[('Aggregate  (cost=528.06..528.07 rows=1 width=8) (actual time=2.789..2.790 rows=1.00 loops=1)',),
 ('  Buffers: shared hit=176',),
 ('  ->  Seq Scan on fait_etablissement  (cost=0.00..523.76 rows=1718 width=4) (actual time=1.240..2.688 rows=1718.00 loops=1)',),
 ("        Filter: (type = 'lycee'::text)",),
 ('        Rows Removed by Filter: 26103',),
 ('        Buffers: shared hit=176',),
 ('Planning:',),
 ('  Buffers: shared hit=28',),
 ('Planning Time: 1.658 ms',),
 ('Execution Time: 2.825 ms',)]

In [21]:
cur.execute("SELECT COUNT(*)  FROM entrepot.fait_etablissement ")
cur.fetchall()

[(27821,)]

In [ ]:
#L'index de clé primaire sert dnas le premier cas car insee_code est la clé primaire; a contrario de type


In [11]:
cur.execute("CREATE INDEX IF NOT EXISTS index_type ON fait_etablissement(type)")

In [20]:
cur.execute("""
SELECT
    indexname,
    indexdef
FROM pg_indexes
WHERE schemaname = 'public'
  AND tablename = %s;
""", ("fait_etablissement",))
cur.fetchall()

[('fait_etablissement_pkey',
  'CREATE UNIQUE INDEX fait_etablissement_pkey ON public.fait_etablissement USING btree (insee_code, type)')]

In [ ]:
#Créer un index sur toutes les colonnes augmenterait grandement la durée d'écriture(et le dépôt est reconstruit régulièrement), et consomme de l'espace disque

In [ ]:
#entrepot.top_communes(p_type text, p_n integer) : les N communes qui ont le plus d'établissements de ce type, avec leur département

In [15]:
cur.execute("DROP FUNCTION IF EXISTS entrepot.top_commune(p_type text, p_n integer) ")
cur.execute("""
    CREATE FUNCTION entrepot.top_commune(p_type text, p_n integer)
    returns TABLE(insee_code VARCHAR(5))
    language plpgsql
    as
    $$  
    begin
        RETURN query
        SELECT fe.insee_code
        FROM fait_etablissement fe
        WHERE type=p_type
        ORDER BY nb
        LIMIT p_n;
    end;        
    $$
    """)

In [ ]:
#entrepot.habitants_par(p_type text) : par département, les habitants, le nombre d'établissements du type, et le ratio habitants par établissement.

In [17]:
cur.execute("DROP FUNCTION IF EXISTS entrepot.habitants_par(p_type text) ")
cur.execute("""
    CREATE FUNCTION entrepot.habitants_par(p_type text)
    returns TABLE(departement TEXT,
                habitants INTEGER,
                nb_etab INTEGER,
                ratio_hab_etab INTEGER
            )
    language plpgsql
    as
    $$
    begin
        RETURN query
        SELECT dc.departement,SUM(population)::INTEGER,SUM(nb)::INTEGER,SUM(population)::INTEGER/SUM(nb)::INTEGER ratio
        FROM fait_etablissement
        JOIN dim_commune dc USING(insee_code)
        WHERE type=p_type
        GROUP BY dc.departement
        ORDER BY dc.departement DESC;
    end;        
    $$
    """)


In [47]:
cur.execute("""SELECT insee_code
        FROM fait_etablissement
        WHERE type='pharmacie'
        ORDER BY nb
        LIMIT 5""")
cur.fetchall()

[('67230',), ('74085',), ('67343',), ('29048',), ('36146',)]

In [18]:
cur.execute("SELECT entrepot.top_commune('pharmacie',5)")
cur.fetchall()

[('67230',), ('74085',), ('67343',), ('29048',), ('36146',)]

In [5]:
cur.execute("SELECT entrepot.habitants_par('pharmacie') ")
cur.fetchall()

[('(Ain,449425,157,2862)',),
 ('(Aisne,311020,165,1884)',),
 ('(Allier,226354,126,1796)',),
 ('(Alpes-de-Haute-Provence,119423,59,2024)',),
 ('(Alpes-Maritimes,1081931,428,2527)',),
 ('(Ardèche,200516,97,2067)',),
 ('(Ardennes,174168,102,1707)',),
 ('(Ariège,87801,48,1829)',),
 ('(Aube,206744,87,2376)',),
 ('(Aude,273000,135,2022)',),
 ('(Aveyron,180763,106,1705)',),
 ('(Bas-Rhin,898593,273,3291)',),
 ('(Bouches-du-Rhône,2068820,733,2822)',),
 ('(Calvados,508059,206,2466)',),
 ('(Cantal,89649,65,1379)',),
 ('(Charente,234502,121,1938)',),
 ('(Charente-Maritime,475546,216,2201)',),
 ('(Cher,205827,99,2079)',),
 ('(Corrèze,165203,92,1795)',),
 ('(Corse-du-Sud,136783,60,2279)',),
 ("(Côte-d'Or,361790,162,2233)",),
 ("(Côtes-d'Armor,442764,183,2419)",),
 ('(Creuse,59515,56,1062)',),
 ('(Deux-Sèvres,264590,119,2223)',),
 ('(Dordogne,253062,138,1833)',),
 ('(Doubs,387228,176,2200)',),
 ('(Drôme,397027,150,2646)',),
 ('(Essonne,1264718,330,3832)',),
 ('(Eure,342067,139,2460)',),
 ('(Eure-et-L

In [65]:
cur.execute("""
    SELECT
        column_name,
        data_type,
        is_nullable,
        column_default
    FROM information_schema.columns
    WHERE table_name = %s
    ORDER BY ordinal_position;
""", ("dim_commune",))
cur.fetchall()

[('insee_code', 'character varying', 'NO', None),
 ('insee_code', 'character varying', 'NO', None),
 ('name', 'text', 'YES', None),
 ('name', 'text', 'YES', None),
 ('departement', 'text', 'YES', None),
 ('departement', 'text', 'YES', None),
 ('region', 'text', 'YES', None),
 ('region', 'text', 'YES', None),
 ('population', 'integer', 'YES', None),
 ('population', 'integer', 'YES', None)]

In [48]:
cur.execute("""CREATE OR REPLACE FUNCTION prevent_delete() 
            RETURNS TRIGGER AS
            $$
            BEGIN 
                RAISE EXCEPTION 'DELETE sur une dimension d entrepot pas autorisé';
            END;
            $$
            LANGUAGE plpgsql;
""")

In [52]:
cur.execute("""
    CREATE OR REPLACE TRIGGER prevent_delete_dim_commune
        BEFORE DELETE
        ON dim_commune
        FOR EACH STATEMENT
        EXECUTE FUNCTION prevent_delete();
""")

In [ ]:
#la maintenance par trigger se justifie si il ya des petites modifications fréquentes sur la base de données et qu'il y a souvent besoin de faire des requêtes sur des 
#données à jour; dans ce cas la reconstruction régulière devient excessive et trop coûteuse

In [ ]:
#Le nombre total de trigger necessaire serait: (nb opérations*nb_tables)-> 3*5=15 

In [72]:
cur.execute("""CREATE OR REPLACE FUNCTION fun_update_warehouse_pharmacie()
            returns TRIGGER
            language plpgsql
            AS
            $$
            BEGIN
                INSERT INTO fait_etablissement(insee_code,type,nb) 
                VALUES(NEW.insee_code,'pharmacie',1)
                ON CONFLICT(insee_code,type) 
                DO UPDATE SET nb=fait_etablissement.nb+1;

                RETURN NEW;
            END;
            $$
            """)

In [73]:
cur.execute("""CREATE OR REPLACE TRIGGER update_warehouse_pharmacie
            AFTER INSERT
            ON pharmacie
            FOR EACH ROW
            EXECUTE FUNCTION fun_update_warehouse_pharmacie();
            """)

In [76]:
cur.execute("INSERT INTO pharmacie(finess,name,insee_code) VALUES (690123460,'test','46252')")

In [77]:
cur.execute("SELECT * FROM fait_etablissement WHERE insee_code = '46252'")
cur.fetchall()

[('46252', 'equipement_sport', 4), ('46252', 'pharmacie', 2)]

In [62]:
cur.execute("SELECT nb,type FROM fait_etablissement WHERE insee_code='69123'")
cur.fetchall()

[(158, 'pharmacie'),
 (57, 'college'),
 (68, 'lycee'),
 (17, 'bibliotheque'),
 (898, 'equipement_sport')]

In [172]:
df_dim_commune=pd.read_sql("SELECT * FROM dim_commune",conn)
df_fait_etablissement=pd.read_sql("SELECT * FROM fait_etablissement",conn)

C:\Users\Utilisateur\AppData\Local\Temp\ipykernel_11112\1915945474.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_dim_commune=pd.read_sql("SELECT * FROM dim_commune",conn)
C:\Users\Utilisateur\AppData\Local\Temp\ipykernel_11112\1915945474.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_fait_etablissement=pd.read_sql("SELECT * FROM fait_etablissement",conn)


In [81]:
df_dim_commune.head()

,insee_code,name,departement,region,population
0,69001,Affoux,Rhône,Auvergne-Rhône-Alpes,398
1,69002,Aigueperse,Rhône,Auvergne-Rhône-Alpes,235
2,69003,Albigny-sur-Saône,Rhône,Auvergne-Rhône-Alpes,3057
3,69004,Alix,Rhône,Auvergne-Rhône-Alpes,777
4,69005,Ambérieux,Rhône,Auvergne-Rhône-Alpes,651


In [82]:
df_fait_etablissement.head()

,insee_code,type,nb
0,67230,pharmacie,1
1,29048,pharmacie,1
2,74085,pharmacie,1
3,67343,pharmacie,1
4,83069,pharmacie,22


In [173]:
df_merged=df_dim_commune.merge(right=df_fait_etablissement,on="insee_code")

In [174]:
df_merged.head()

,insee_code,name,departement,region,population,type,nb
0,69001,Affoux,Rhône,Auvergne-Rhône-Alpes,398,equipement_sport,1
1,69002,Aigueperse,Rhône,Auvergne-Rhône-Alpes,235,bibliotheque,1
2,69002,Aigueperse,Rhône,Auvergne-Rhône-Alpes,235,equipement_sport,4
3,69003,Albigny-sur-Saône,Rhône,Auvergne-Rhône-Alpes,3057,pharmacie,1
4,69003,Albigny-sur-Saône,Rhône,Auvergne-Rhône-Alpes,3057,bibliotheque,1


In [ ]:
df_merged

In [175]:
df_merged_by_dep_type=df_merged[["population","nb","region","type","departement"]].groupby(by=["departement","type","region"]).sum()
df_merged_by_dep_type.head()

population    nb
departement type             region                                
Ain         bibliotheque     Auvergne-Rhône-Alpes      564735   230
            college          Auvergne-Rhône-Alpes      327171    78
            equipement_sport Auvergne-Rhône-Alpes      675660  4105
            lycee            Auvergne-Rhône-Alpes      177529    43
            pharmacie        Auvergne-Rhône-Alpes      449425   157

In [107]:
df_merged_by_dep_type

population    nb
departement type             region                                
Ain         bibliotheque     Auvergne-Rhône-Alpes      564735   230
            college          Auvergne-Rhône-Alpes      327171    78
            equipement_sport Auvergne-Rhône-Alpes      675660  4105
            lycee            Auvergne-Rhône-Alpes      177529    43
            pharmacie        Auvergne-Rhône-Alpes      449425   157
...                                                       ...   ...
Yvelines    bibliotheque     Île-de-France            1420379   197
            college          Île-de-France            1227769   188
            equipement_sport Île-de-France            1477221  4691
            lycee            Île-de-France             892207   116
            pharmacie        Île-de-France            1370201   370

[505 rows x 2 columns]

In [176]:
df_merged_by_dep_type=df_merged_by_dep_type.reset_index()

In [110]:
df_merged_by_dep_type

,departement,type,region,population,nb
0,Ain,bibliotheque,Auvergne-Rhône-Alpes,564735,230
1,Ain,college,Auvergne-Rhône-Alpes,327171,78
2,Ain,equipement_sport,Auvergne-Rhône-Alpes,675660,4105
3,Ain,lycee,Auvergne-Rhône-Alpes,177529,43
4,Ain,pharmacie,Auvergne-Rhône-Alpes,449425,157
...,...,...,...,...,...
500,Yvelines,bibliotheque,Île-de-France,1420379,197
501,Yvelines,college,Île-de-France,1227769,188
502,Yvelines,equipement_sport,Île-de-France,1477221,4691
503,Yvelines,lycee,Île-de-France,892207,116


In [177]:
df_pivoted=df_merged_by_dep_type.pivot(index="departement",values="nb",columns="type")
df_pivoted.head()

type,bibliotheque,college,equipement_sport,lycee,pharmacie
departement,,,,,
Ain,230,78,4105,43,157
Aisne,129,92,3638,51,165
Allier,215,51,2452,32,126
Alpes-Maritimes,126,122,3592,79,428
Alpes-de-Haute-Provence,94,23,1815,16,59


In [121]:
df_pivoted.columns

Index(['bibliotheque', 'college', 'equipement_sport', 'lycee', 'pharmacie'], dtype='str', name='type')

In [136]:
df_pivoted.index

Index(['Ain', 'Aisne', 'Allier', 'Alpes-Maritimes', 'Alpes-de-Haute-Provence',
       'Ardennes', 'Ardèche', 'Ariège', 'Aube', 'Aude',
       ...
       'Territoire de Belfort', 'Val-d'Oise', 'Val-de-Marne', 'Var',
       'Vaucluse', 'Vendée', 'Vienne', 'Vosges', 'Yonne', 'Yvelines'],
      dtype='str', name='departement', length=101)

In [178]:
df_pivoted.columns=df_pivoted.columns.set_names("nb_per_type")

In [179]:
df_pivoted

nb_per_type,bibliotheque,college,equipement_sport,lycee,pharmacie
departement,,,,,
Ain,230,78,4105,43,157
Aisne,129,92,3638,51,165
Allier,215,51,2452,32,126
Alpes-Maritimes,126,122,3592,79,428
Alpes-de-Haute-Provence,94,23,1815,16,59
...,...,...,...,...,...
Vendée,236,88,4074,58,209
Vienne,191,60,2602,35,138
Vosges,134,61,2731,39,126


In [169]:
query="""SELECT d0.name,
            (SELECT COUNT(*) FROM bibliotheque b 
            JOIN commune c USING(insee_code) 
            WHERE c.code_departement= d0.code_departement
            GROUP BY code_departement) 
            bibliotheque,
            (SELECT COUNT(*) FROM college co
            JOIN commune c USING(insee_code) 
            WHERE c.code_departement= d0.code_departement
            GROUP BY code_departement) 
            college,
            (SELECT COUNT(*) FROM equipement_sport eq
            JOIN commune c USING(insee_code) 
            WHERE c.code_departement= d0.code_departement
            GROUP BY code_departement) 
            equipement_sport,
            (SELECT COUNT(*) FROM lycee l
            JOIN commune c USING(insee_code) 
            WHERE c.code_departement= d0.code_departement
            GROUP BY code_departement) 
            lycee,
            (SELECT COUNT(*) FROM pharmacie p
            JOIN commune c USING(insee_code) 
            WHERE c.code_departement= d0.code_departement
            GROUP BY code_departement) 
            pharmacie
            FROM departement d0
            ORDER BY d0.name COLLATE "C"
            """
cur.execute(query)
cur.fetchall()

[('Ain', 230, 78, 4105, 43, 157),
 ('Aisne', 129, 92, 3638, 51, 165),
 ('Allier', 215, 51, 2452, 32, 126),
 ('Alpes-Maritimes', 126, 122, 3592, 79, 428),
 ('Alpes-de-Haute-Provence', 94, 23, 1815, 16, 59),
 ('Ardennes', 100, 49, 1742, 24, 102),
 ('Ardèche', 218, 47, 2456, 24, 97),
 ('Ariège', 76, 23, 1896, 18, 48),
 ('Aube', 144, 43, 1497, 26, 87),
 ('Aude', 251, 44, 2946, 35, 135),
 ('Aveyron', 187, 47, 3068, 33, 106),
 ('Bas-Rhin', 215, 144, 4718, 86, 273),
 ('Bouches-du-Rhône', 141, 242, 5877, 174, 733),
 ('Calvados', 131, 88, 3644, 61, 206),
 ('Cantal', 153, 29, 1701, 17, 65),
 ('Charente', 75, 59, 1737, 30, 121),
 ('Charente-Maritime', 224, 76, 4092, 44, 216),
 ('Cher', 148, 44, 2174, 27, 99),
 ('Corrèze', 117, 36, 2288, 30, 92),
 ('Corse-du-Sud', 49, 20, 555, 12, 60),
 ('Creuse', 98, 24, 1137, 11, 56),
 ("Côte-d'Or", 208, 73, 3053, 38, 162),
 ("Côtes-d'Armor", 259, 90, 4236, 50, 183),
 ('Deux-Sèvres', 141, 63, 2468, 29, 119),
 ('Dordogne', 189, 65, 2709, 32, 138),
 ('Doubs', 193,

In [170]:
df_query=pd.read_sql(query,conn,index_col="name")

C:\Users\Utilisateur\AppData\Local\Temp\ipykernel_11112\2064237757.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_query=pd.read_sql(query,conn,index_col="name")


In [181]:
df_query.columns=df_query.columns.set_names("nb_per_type")
df_query.index=df_query.index.set_names("departement")

In [166]:
df_query[0:20]

nb_per_type,bibliotheque,college,equipement_sport,lycee,pharmacie
departement,,,,,
Ain,230,78,4105,43,157
Aisne,129,92,3638,51,165
Allier,215,51,2452,32,126
Alpes-Maritimes,126,122,3592,79,428
Alpes-de-Haute-Provence,94,23,1815,16,59
Ardennes,100,49,1742,24,102
Ardèche,218,47,2456,24,97
Ariège,76,23,1896,18,48
Aube,144,43,1497,26,87


In [167]:
df_pivoted[0:20]

nb_per_type,bibliotheque,college,equipement_sport,lycee,pharmacie
departement,,,,,
Ain,230,78,4105,43,157
Aisne,129,92,3638,51,165
Allier,215,51,2452,32,126
Alpes-Maritimes,126,122,3592,79,428
Alpes-de-Haute-Provence,94,23,1815,16,59
Ardennes,100,49,1742,24,102
Ardèche,218,47,2456,24,97
Ariège,76,23,1896,18,48
Aube,144,43,1497,26,87


In [ ]:
pd.testing.assert_frame_equal(df_query,df_pivoted)